In [ ]:
import json
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime, timezone
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ess').is_dir() and (p / 'data/cstr').is_dir()), None)  # Repository root, independent of the notebook working directory.
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from ess.models import create_model, ControllerInput
from ess.data import load_trajectory, FeatureHistory
from ess.processes import CSTRPlant
from ess.reproducibility import seed_everything
import torch


In [ ]:
RUN_DIR = None  # None selects the newest completed run; alternatively set Path('outputs/training/<run_name>') or an absolute path.
CHECKPOINT_NAME = 'selected_model.pkl'  # Eligible selected model; final_ess.pkl evaluates the last ESS epoch instead.
DEVICE = 'cpu'  # Inference device; CPU is sufficient for evaluation.
PROGRESS_EVERY = 2000  # Samples between progress messages; 0 disables intermediate messages.
SAVE_RESULTS = True  # Save metrics, trajectories and figures under outputs/evaluation.
if RUN_DIR is None:
    completed_runs = sorted((p for p in (REPO_ROOT / 'outputs/training').glob('ess_cstr_*') if (p / 'selected_model.pkl').is_file() and (p / 'selection.json').is_file()), key=lambda p: p.name)
    if not completed_runs:
        raise FileNotFoundError('No completed training run found. Run 01_train_cstr.ipynb first, or evaluate the bundled checkpoint with 03_evaluate_paper_model.ipynb.')
    RUN_DIR = completed_runs[-1]
else:
    RUN_DIR = Path(RUN_DIR)
    if not RUN_DIR.is_absolute():
        RUN_DIR = REPO_ROOT / RUN_DIR
RUN_NAME = RUN_DIR.name  # Run identifier recorded with evaluation results.
OUTPUT_DIR = REPO_ROOT / 'outputs/evaluation' / RUN_NAME / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')  # Separate evaluation results for each invocation.


In [ ]:
def autonomous_rollout(model, trajectory, config, input_mean, input_scale, output_mean, output_scale, device, label):
    """Run the ANN alone with frozen weights and fresh plant/history state.

    At sample zero the applied action is zero. Each subsequent interval uses
    the preceding ANN prediction. There is no PID, sampling, blending or noise.
    Timing, input windows and process dynamics match the ESS evaluation path.
    """
    plant = CSTRPlant(config.sample_time_min)
    history = FeatureHistory(config)
    controller = ControllerInput(config.architecture, config.lstm_window, device)
    rows = np.empty((len(trajectory.reference), 8), dtype=float)
    action = 0.0
    model.eval()
    with torch.no_grad():
        for i, reference in enumerate(trajectory.reference):
            measurement = plant.step(action, trajectory.dq[i], trajectory.dcai[i])
            error = float(reference) - measurement
            features = history.update(error, action)
            prediction = controller.predict(model, (features - input_mean) / input_scale)
            next_action = prediction.item() * output_scale + output_mean
            rows[i] = [i * config.sample_time_min, reference, measurement, error,
                       action, next_action, trajectory.dcai[i], trajectory.dq[i]]
            if not np.isfinite(rows[i]).all():
                raise RuntimeError(f'{label}: non-finite trajectory at sample {i}; no finite IAE can be reported.')
            action = next_action
            if PROGRESS_EVERY and (i + 1) % PROGRESS_EVERY == 0:
                print(f'{label}: {i + 1}/{len(rows)} samples', flush=True)
    absolute_error = np.abs(rows[:, 3])
    cumulative_iae = cumulative_trapezoid(absolute_error, rows[:, 0], initial=0)
    metrics = dict(dataset=label, samples=len(rows), duration_min=float(rows[-1, 0]),
                   iae_trapezoid=float(cumulative_iae[-1]),
                   iae_rectangular_all_samples=float(config.sample_time_min * absolute_error.sum()),
                   iae_units='process-signal units * min')
    return rows, cumulative_iae, metrics


def plot_evaluation(rows, cumulative_iae, metrics):
    """Display process tracking, applied action and accumulated physical tracking IAE."""
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    time = rows[:, 0]
    axes[0].plot(time, rows[:, 1], label='Reference', color='black', linestyle='--')
    axes[0].plot(time, rows[:, 2], label='ANN-controlled process')
    axes[0].set_ylabel('Process signal')
    axes[1].plot(time, rows[:, 4], label='Applied ANN action', color='tab:green')
    axes[1].set_ylabel('Action (physical units)')
    axes[2].plot(time, cumulative_iae, label='Cumulative IAE (trapezoidal)', color='tab:red')
    axes[2].set(ylabel='IAE (signal units·min)', xlabel='Time (min)')
    for axis in axes:
        axis.legend()
        axis.grid(True, alpha=.3)
    fig.suptitle(f"{metrics['dataset'].title()} trajectory — ANN alone — IAE = {metrics['iae_trapezoid']:.6f}")
    fig.tight_layout(rect=(0, 0, 1, .96))
    if SAVE_RESULTS:
        fig.savefig(OUTPUT_DIR / f"{metrics['dataset']}_tracking.png", dpi=150)
    plt.show()
    plt.close(fig)


In [ ]:
saved_config = json.loads((RUN_DIR / 'config.json').read_text())
config = SimpleNamespace(**saved_config)
selection = json.loads((RUN_DIR / 'selection.json').read_text())
input_mean = np.loadtxt(RUN_DIR / 'scalers' / 'input_mean.csv').reshape(4)
input_scale = np.loadtxt(RUN_DIR / 'scalers' / 'input_scale.csv').reshape(4)
output_mean = float(np.loadtxt(RUN_DIR / 'scalers' / 'output_mean.csv'))
output_scale = float(np.loadtxt(RUN_DIR / 'scalers' / 'output_scale.csv'))
seed_everything(config.seed, config.deterministic)
device = torch.device(DEVICE)
model = create_model(config.architecture, device)
model.load_state_dict(torch.load(RUN_DIR / CHECKPOINT_NAME, map_location=device, weights_only=True))
model.requires_grad_(False)
model.eval()
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(f'Run: {RUN_NAME}\nCheckpoint: {CHECKPOINT_NAME}\nArchitecture: {config.architecture}')
if CHECKPOINT_NAME == 'selected_model.pkl':
    print(f"Selected {selection['selected']['stage']} epoch: {selection['selected']['epoch']}")
print(f'Sample interval: {config.sample_time_min} min; saved dataset fraction: {config.dataset_fraction}')
print('ANN alone; frozen model and saved scalers; no PID control, blending or noise.')


In [ ]:
results = {}
metrics_rows = []
data_dir = REPO_ROOT / config.data_dir
for label, data_file, excitation_file in (
    ('training', config.training_file, config.training_excitation),
    ('validation', config.validation_file, config.validation_excitation),
):
    trajectory = load_trajectory(data_dir / data_file, data_dir / excitation_file, config)
    print(f'Running {label}: {len(trajectory.reference)} samples...', flush=True)
    rows, cumulative_iae, metrics = autonomous_rollout(
        model, trajectory, config, input_mean, input_scale, output_mean, output_scale, device, label)
    results[label] = (rows, cumulative_iae)
    metrics_rows.append(metrics)
    print(f"{label}: IAE (trapezoid) = {metrics['iae_trapezoid']:.6f}; "
          f"IAE (dt × sum) = {metrics['iae_rectangular_all_samples']:.6f}", flush=True)
    if SAVE_RESULTS:
        np.savetxt(OUTPUT_DIR / f'{label}_rollout.csv', np.column_stack((rows, cumulative_iae)),
                   delimiter=',', comments='',
                   header='time_min,reference,process_output,error,applied_action,ann_action_next,dCAi,dQ,cumulative_iae')
    plot_evaluation(rows, cumulative_iae, metrics)


In [ ]:
print('\nANN-only tracking IAE — signal units·minutes')
print(f"{'Dataset':<14} {'Samples':>8} {'Duration (min)':>16} {'IAE trapezoid':>16} {'IAE dt*sum':>16}")
for metric in metrics_rows:
    print(f"{metric['dataset']:<14} {metric['samples']:>8} {metric['duration_min']:>16.2f} "
          f"{metric['iae_trapezoid']:>16.6f} {metric['iae_rectangular_all_samples']:>16.6f}")
print('\nPaper Table II, ESS LeakyMLP4 (308e): training IAE 345; validation IAE 272.')
print('Those are published reference values, not expected exact results for this retrained checkpoint.')
print('This evaluates both existing trajectories; the training trajectory is not an independent test set.')
if SAVE_RESULTS:
    report = dict(run=RUN_NAME, checkpoint=CHECKPOINT_NAME, device=str(device),
                  selected_checkpoint_metadata=selection['selected'] if CHECKPOINT_NAME == 'selected_model.pkl' else None,
                  sample_time_min=config.sample_time_min, dataset_fraction=config.dataset_fraction,
                  metrics=metrics_rows, saved_training_config=saved_config)
    (OUTPUT_DIR / 'metrics.json').write_text(json.dumps(report, indent=2) + '\n')
    import csv
    with (OUTPUT_DIR / 'metrics.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(metrics_rows[0]))
        writer.writeheader()
        writer.writerows(metrics_rows)
    print(f'\nEvaluation files: {OUTPUT_DIR}')
